<a href="https://colab.research.google.com/github/nyp-sit/iti121-2026s2/blob/main/L5/2_optuna_pytorch_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hyperparameter Tuning with Optuna and PyTorch

This practical demonstrates how to perform **hyperparameter tuning** using **Optuna** with a **PyTorch** neural network.

1. Load and prepare the dataset  
2. Define a tunable neural network  
3. Define the hyperparameter search space  
4. Run the tuner  
5. Inspect the best hyperparameters  
6. Build and train the best model  
7. Evaluate the final model  

> The dataset is downloaded automatically. No external data file is required.


## 1. Install Optuna (if needed)

In [ ]:
# Uncomment this line if Optuna is not installed
!pip install optuna

## 2. Import Libraries

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam, SGD

import torchvision
from torchvision import transforms

import optuna

import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("Optuna version:", optuna.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 3. Load the Dataset

The Fashion-MNIST dataset contains 28 × 28 grayscale images from 10 clothing categories.

`download=True` downloads the dataset automatically the first time the notebook is run.


In [ ]:
transform = transforms.ToTensor()

full_train_set = torchvision.datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_set = torchvision.datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Split the original training set into training and validation sets
train_size = int(0.8 * len(full_train_set))
valid_size = len(full_train_set) - train_size

generator = torch.Generator().manual_seed(42)

train_set, valid_set = random_split(
    full_train_set,
    [train_size, valid_size],
    generator=generator
)

print("Training samples:", len(train_set))
print("Validation samples:", len(valid_set))
print("Test samples:", len(test_set))


## 4. Define the Tunable PyTorch Model


With Optuna, hyperparameters are commonly defined using functions such as:

```python
trial.suggest_int(...)
trial.suggest_categorical(...)
trial.suggest_float(...)
```

The following model allows Optuna to tune:

- number of hidden units
- dropout rate


In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, hidden_units=128, dropout_rate=0.0):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_units),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_units, 10)
        )

    def forward(self, x):
        return self.network(x)


## 5. Training and Validation Functions

In [ ]:
loss_function = nn.CrossEntropyLoss()


def train_one_epoch(model, train_loader, optimizer):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        output = model(x)
        loss = loss_function(output, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        pred = output.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return total_loss / len(train_loader), correct / total


def validate(model, valid_loader):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)

            output = model(x)
            loss = loss_function(output, y)

            total_loss += loss.item()

            pred = output.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    return total_loss / len(valid_loader), correct / total


## 6. Define the Optuna Objective

An Optuna **trial** represents one combination of hyperparameters.

For each trial, Optuna will:

1. choose hyperparameter values
2. build a PyTorch model
3. train the model
4. measure validation accuracy
5. use the result to decide what to try next


In [ ]:
def objective(trial):

    # Hyperparameters to tune
    hidden_units = trial.suggest_int(
        "hidden_units",
        32,
        512,
        step=32
    )

    learning_rate = trial.suggest_categorical(
        "learning_rate",
        [1e-2, 1e-3, 1e-4]
    )

    dropout_rate = trial.suggest_float(
        "dropout_rate",
        0.0,
        0.5,
        step=0.1
    )

    batch_size = 32

    # Data loaders can depend on the selected batch size
    train_loader = DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=True
    )

    valid_loader = DataLoader(
        valid_set,
        batch_size=batch_size,
        shuffle=False
    )

    # Build a new model for this trial
    model = NeuralNetwork(
        hidden_units=hidden_units,
        dropout_rate=dropout_rate
    ).to(device)


    optimizer = Adam(model.parameters(), lr=learning_rate)

    # Keep tuning trials short
    num_epochs = 5

    for epoch in range(num_epochs):

        train_loss, train_accuracy = train_one_epoch(
            model,
            train_loader,
            optimizer
        )

        valid_loss, valid_accuracy = validate(
            model,
            valid_loader
        )

        # Send intermediate result to Optuna
        trial.report(valid_accuracy, epoch)

        # Stop poor trials early
        if trial.should_prune():
            raise optuna.TrialPruned()

    return valid_accuracy


## 7. Run the Hyperparameter Search

`direction="maximize"` tells Optuna that a higher validation accuracy is better.

For a classroom practical, a small number of trials keeps runtime manageable. Increase `n_trials` for a more thorough search.


In [ ]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=15
)


## 8. View the Best Hyperparameters

In [ ]:
print("Best validation accuracy:", study.best_value)
print("\nBest hyperparameters:")

for key, value in study.best_params.items():
    print(f"{key}: {value}")


The best trial can also be inspected directly:


In [ ]:
best_trial = study.best_trial

print("Trial number:", best_trial.number)
print("Validation accuracy:", best_trial.value)
print("Parameters:", best_trial.params)


## 9. Train the Best Model

After hyperparameter tuning, we create a new model using the best hyperparameters and train it for more epochs.


In [ ]:
best_params = study.best_params

train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True
)

valid_loader = DataLoader(
    valid_set,
    batch_size=32,
    shuffle=False
)

best_model = NeuralNetwork(
    hidden_units=best_params["hidden_units"],
    dropout_rate=best_params["dropout_rate"]
).to(device)


best_optimizer = Adam(
    best_model.parameters(),
    lr=best_params["learning_rate"]
)

num_epochs = 10

train_loss_history = []
valid_loss_history = []
train_acc_history = []
valid_acc_history = []

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        best_model,
        train_loader,
        best_optimizer
    )

    valid_loss, valid_acc = validate(
        best_model,
        valid_loader
    )

    train_loss_history.append(train_loss)
    valid_loss_history.append(valid_loss)
    train_acc_history.append(train_acc)
    valid_acc_history.append(valid_acc)

    print(
        f"Epoch {epoch + 1:2d} | "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} | "
        f"Valid Loss: {valid_loss:.4f} "
        f"Valid Acc: {valid_acc:.4f}"
    )


## 10. Plot Training and Validation Performance

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(train_loss_history, label="Training Loss")
plt.plot(valid_loss_history, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(train_acc_history, label="Training Accuracy")
plt.plot(valid_acc_history, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


## 11. Evaluate the Best Model on the Test Set

In [ ]:
test_loader = DataLoader(
    test_set,
    batch_size=32,
    shuffle=False
)

test_loss, test_accuracy = validate(
    best_model,
    test_loader
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


## Key idea

**define search space → train candidate models → evaluate validation performance → select the best hyperparameters**

Optuna is framework-independent, so it works naturally with PyTorch as well as many other machine-learning frameworks.
